<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.transforms import Resize, CenterCrop, ToTensor, Normalize, Compose
from PIL import Image

# 1. Clone the repository from your screenshot
REPO_DIR = "imagenet-sample-images"
REPO_URL = "https://github.com/EliSchwartz/imagenet-sample-images.git"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print("ImageNet sample repository already exists.")

# 2. Define the Split Execution Methods

def forward_baseline(model, x):
    return model(x)

def forward_disnet(model, x):
    """
    DISNET (Lossless): Splits the input horizontally into 2 halves,
    including a 1-row overlap (halo) to preserve boundary math.
    """
    B, C, H, W = x.shape
    mid = H // 2

    x1 = x[:, :, 0:mid+1, :]
    x2 = x[:, :, mid-1:H, :]

    conv1 = model.features[0]

    x1_padded = F.pad(x1, (1, 1, 1, 0))
    out1 = F.conv2d(x1_padded, conv1.weight, conv1.bias, stride=conv1.stride)

    x2_padded = F.pad(x2, (1, 1, 0, 1))
    out2 = F.conv2d(x2_padded, conv1.weight, conv1.bias, stride=conv1.stride)

    merged_features = torch.cat([out1, out2], dim=2)

    x_rest = merged_features
    for layer in model.features[1:]:
        x_rest = layer(x_rest)

    x_rest = model.avgpool(x_rest)
    x_rest = torch.flatten(x_rest, 1)
    logits = model.classifier(x_rest)
    return logits

def forward_naive(model, x):
    """
    Naive / DeepSlicing (Lossy): Splits the input horizontally into 2 halves
    WITHOUT overlap, causing zero-padding artifacts at the split boundary.
    """
    B, C, H, W = x.shape
    mid = H // 2

    x1 = x[:, :, 0:mid, :]
    x2 = x[:, :, mid:H, :]

    conv1 = model.features[0]

    x1_padded = F.pad(x1, (1, 1, 1, 1))
    out1 = F.conv2d(x1_padded, conv1.weight, conv1.bias, stride=conv1.stride)

    x2_padded = F.pad(x2, (1, 1, 1, 1))
    out2 = F.conv2d(x2_padded, conv1.weight, conv1.bias, stride=conv1.stride)

    merged_features = torch.cat([out1, out2], dim=2)

    x_rest = merged_features
    for layer in model.features[1:]:
        x_rest = layer(x_rest)

    x_rest = model.avgpool(x_rest)
    x_rest = torch.flatten(x_rest, 1)
    logits = model.classifier(x_rest)
    return logits

# 3. Self-Contained ImageNet Loader
def load_imagenet_samples(k=20):
    """Reconstructs the 0-999 class mapping from local filenames and loads k random samples."""
    # Get all image files from the cloned repository
    all_files = [f for f in os.listdir(REPO_DIR) if f.endswith(".JPEG")]

    if len(all_files) == 0:
        raise FileNotFoundError(f"No .JPEG files found in {REPO_DIR}. Please check the repository clone.")

    # Extract unique WordNet IDs (wnids) from the filenames
    # e.g., "n01440764_tench.JPEG" -> "n01440764"
    unique_wnids = sorted(list(set(f.split("_")[0] for f in all_files)))

    # Build the mapping: alphabetical order of wnids corresponds to PyTorch's 0-999 indices
    wnid_to_idx = {wnid: idx for idx, wnid in enumerate(unique_wnids)}

    # Select k random files for evaluation
    selected_files = random.sample(all_files, min(k, len(all_files)))

    transform = Compose([
        Resize(224),
        CenterCrop(224),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    images = []
    targets = []
    names = []

    for filename in selected_files:
        parts = filename.replace(".JPEG", "").split("_")
        wnid = parts[0]
        class_name = " ".join(parts[1:])

        class_idx = wnid_to_idx[wnid]
        img_path = os.path.join(REPO_DIR, filename)
        try:
            img = Image.open(img_path).convert('RGB')
            images.append(transform(img))
            targets.append(class_idx)
            names.append(class_name)
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    return torch.stack(images), torch.tensor(targets), names

# 4. Main Evaluation Pipeline
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running evaluation on: {device}\n")

    # Load Pretrained VGG16
    print("Loading pretrained VGG16 model...")
    vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(device).eval()

    # Load k random ImageNet samples
    k_samples = 30
    print(f"\nLoading {k_samples} random ImageNet samples from local repository...")
    images, targets, names = load_imagenet_samples(k=k_samples)
    images, targets = images.to(device), targets.to(device)

    # Run Inference
    print("\nRunning inference...")
    with torch.no_grad():
        out_base = forward_baseline(vgg16, images)
        out_disnet = forward_disnet(vgg16, images)
        out_naive = forward_naive(vgg16, images)

    # Verify mathematical equivalence of DISNET
    disnet_matches = torch.allclose(out_base, out_disnet, atol=1e-4)
    print(f"DISNET mathematically matches Baseline: {disnet_matches}")

    # Calculate Accuracies
    correct_base = 0
    correct_disnet = 0
    correct_naive = 0

    for i in range(len(names)):
        target_label = targets[i].item()

        pred_base = out_base[i].argmax().item()
        pred_disnet = out_disnet[i].argmax().item()
        pred_naive = out_naive[i].argmax().item()

        if pred_base == target_label: correct_base += 1
        if pred_disnet == target_label: correct_disnet += 1
        if pred_naive == target_label: correct_naive += 1

    # Print final accuracies
    total = len(names)
    print("\n" + "="*50)
    print(f"EVALUATION RESULTS ON {total} IMAGENET SAMPLES")
    print("="*50)
    print(f"Baseline Accuracy: {correct_base/total*100:.2f}%")
    print(f"DISNET Accuracy:   {correct_disnet/total*100:.2f}%")
    print(f"Naive Accuracy:    {correct_naive/total*100:.2f}%")
    print("="*50)

if __name__ == "__main__":
    main()

ImageNet sample repository already exists.
Running evaluation on: cpu

Loading pretrained VGG16 model...

Loading 30 random ImageNet samples from local repository...

Running inference...
DISNET mathematically matches Baseline: True

EVALUATION RESULTS ON 30 IMAGENET SAMPLES
Baseline Accuracy: 93.33%
DISNET Accuracy:   93.33%
Naive Accuracy:    90.00%
